In [ ]:
# Cell 1 – Install & Imports

!pip install -q datasets tqdm

import json
import random
import hashlib
import logging
import unicodedata
from pathlib import Path
from typing import Optional, Dict, Tuple
from collections import defaultdict
from tqdm.auto import tqdm

In [ ]:
# Cell 2 – Validate → Strong Mandarin Filter 

# ====================== PATHS ======================
INPUT_FILE = Path("/kaggle/input/datasets/abdighaz/en-zh-finetune/finetune_enzh_train.jsonl")
OUTPUT_DIR = Path("/kaggle/working/splits")
LOGS = Path("/kaggle/working/logs")
MANIFESTS = Path("/kaggle/working/manifests")

LOG_PATH = LOGS / "split_pipeline.log"

# ====================== CONFIG ======================
CONFIG = {
    "drop_identical_source_target": True,
    "require_chinese_on_zh_side": True,
    "length_thresholds": {
        "min_chars_source": 15,
        "min_chars_target": 8,
        "max_chars_source": 450,
        "max_chars_target": 450
    }
}

KEEP_PERCENT = {
    "Helsinki-NLP/news_commentary": 1.0,
    "Helsinki-NLP/un_pc": 0.04,
    "larryvrh/WikiMatrix-v1-En_Zh-filtered": 0.7,
    "ymoslem/Tatoeba-Translations": 1.0
}

SPLIT_SIZES = {
    "finetune": 20_000,
    "judge_calibration": 1_000,
    "prompt_validation": 1_000,
    "sealed_final_test": 1_000
}

RANDOM_SEED = 42

# ====================== LOGGING ======================
LOGS.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTS.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_PATH, encoding="utf-8"),
        logging.StreamHandler()
    ]
)

# ====================== UTILS ======================
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def normalize_text(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None
    text = unicodedata.normalize("NFKC", str(text))
    text = text.replace("\u00a0", " ").strip()
    text = " ".join(text.split())
    return text

def utf8_readable(text: str) -> bool:
    try:
        text.encode("utf-8").decode("utf-8")
        return True
    except:
        return False

def contains_cjk(text: str) -> bool:
    return any('\u4e00' <= ch <= '\u9fff' for ch in text)

# Stronger Mandarin + Simplified Chinese filter
def is_simplified_mandarin(text: str) -> bool:
    if not contains_cjk(text):
        return False

    # Common Traditional-only characters (reject if too many appear)
    traditional_chars = set("國語們說時後還這來對麼嗎為會學業發經東西門開閉見覺體機車書長")
    trad_count = sum(1 for c in text if c in traditional_chars)

    # Allow a very small number of traditional characters (OCR noise, etc.)
    if trad_count > 2:
        return False

    return True

def basic_pair_filter(src: str, tgt: str) -> bool:
    if not src or not tgt:
        return False
    if CONFIG["drop_identical_source_target"] and src == tgt:
        return False

    lt = CONFIG["length_thresholds"]
    if not (lt["min_chars_source"] <= len(src) <= lt["max_chars_source"]):
        return False
    if not (lt["min_chars_target"] <= len(tgt) <= lt["max_chars_target"]):
        return False

    if not utf8_readable(src) or not utf8_readable(tgt):
        return False

    if CONFIG["require_chinese_on_zh_side"] and not contains_cjk(tgt):
        return False

    # Strong Mandarin filter
    if not is_simplified_mandarin(tgt):
        return False

    return True

def validate_record(rec: Dict) -> Tuple[bool, str]:
    required_fields = [
        "id", "pair_id", "source_text", "target_text",
        "source_lang", "target_lang", "domain", "license", "dataset_name"
    ]

    for f in required_fields:
        if f not in rec or rec[f] is None:
            return False, f"missing_or_null:{f}"
        if not isinstance(rec[f], str) or not rec[f].strip():
            return False, f"invalid_string:{f}"

    if rec["source_lang"] != "en":
        return False, "source_lang_not_en"
    if rec["target_lang"] != "zh":
        return False, "target_lang_not_zh"

    if not utf8_readable(rec["source_text"]) or not utf8_readable(rec["target_text"]):
        return False, "utf8_error"

    return True, "ok"

In [ ]:
# Cell 3 – Full Pipeline 

# =================================================================
# Full Pipeline: Validate → Strong Mandarin Filter → Sample → Split
# =================================================================

# ====================== MAIN PROCESS =============================
random.seed(RANDOM_SEED)
logging.info("Starting pipeline with strong Mandarin filter...")

data_by_dataset = defaultdict(list)
stats = {"total_read": 0, "passed": 0, "failed": 0, "fail_reasons": defaultdict(int)}

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Validating + Mandarin filtering"):
        stats["total_read"] += 1
        try:
            item = json.loads(line)
        except:
            stats["failed"] += 1
            stats["fail_reasons"]["json_error"] += 1
            continue

        item["source_text"] = normalize_text(item.get("source_text"))
        item["target_text"] = normalize_text(item.get("target_text"))

        ok, reason = validate_record(item)
        if not ok:
            stats["failed"] += 1
            stats["fail_reasons"][reason] += 1
            continue

        if not basic_pair_filter(item["source_text"], item["target_text"]):
            stats["failed"] += 1
            stats["fail_reasons"]["basic_or_mandarin_filter"] += 1
            continue

        data_by_dataset[item["dataset_name"]].append(item)
        stats["passed"] += 1

logging.info(f"Total read : {stats['total_read']:,}")
logging.info(f"Passed     : {stats['passed']:,}")
logging.info(f"Failed     : {stats['failed']:,}")
for k, v in stats["fail_reasons"].items():
    logging.info(f"  - {k}: {v}")

# ----------------- Sample by percentage --------------------
sampled_data = []
logging.info("Sampling by percentage...")

for dataset_name, items in data_by_dataset.items():
    percent = KEEP_PERCENT.get(dataset_name, 1.0)
    keep_n = int(len(items) * percent)
    random.shuffle(items)
    selected = items[:keep_n]
    sampled_data.extend(selected)
    logging.info(f"{dataset_name}: kept {keep_n:,} / {len(items):,} ({percent*100:.0f}%)")

logging.info(f"Total after sampling: {len(sampled_data):,}")
random.shuffle(sampled_data)

# ----------------- Create splits --------------------------
total_needed = sum(SPLIT_SIZES.values())
if len(sampled_data) < total_needed:
    raise ValueError(f"Not enough clean data! Have {len(sampled_data):,}, need {total_needed:,}")

splits = {}
start = 0
for name, size in SPLIT_SIZES.items():
    end = start + size
    splits[name] = sampled_data[start:end]
    start = end
    logging.info(f"{name:20}: {len(splits[name]):,}")

# -------------------- Save ------------------------------
for name, items in splits.items():
    out_path = OUTPUT_DIR / f"{name}.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    logging.info(f"Saved {out_path} | sha256={sha256_file(out_path)}")

# Manifest
manifest = {
    "split_unique_pairs": {k: len(v) for k, v in splits.items()},
    "keep_percent_per_dataset": KEEP_PERCENT,
    "length_thresholds": CONFIG["length_thresholds"],
    "random_seed": RANDOM_SEED,
    "validation_stats": {
        "total_read": stats["total_read"],
        "passed": stats["passed"],
        "failed": stats["failed"],
        "fail_reasons": dict(stats["fail_reasons"])
    },
    "file_hashes": {
        name: sha256_file(OUTPUT_DIR / f"{name}.jsonl") for name in splits
    }
}

manifest_path = MANIFESTS / "split_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

logging.info(f"Manifest saved → {manifest_path}")
logging.info("Pipeline finished successfully!")